In [9]:

"""
Trichoderma atroviride promoter extraction
===========================================

Genome:
    Trichoderma atroviride P1
    Assembly: GCF_020647795.1
    Chromosome: NC_089400.1

Run directly from VS Code:
    Press "Run Python File"

No command-line arguments are required.

Required files:
    1. NC_089400.1.fasta
    2. annotation.gff3

Place both files in the same folder as this script.
"""


# =========================================================
# USER SETTINGS
# =========================================================

# ---------------------------------------------------------
# INPUT FILES
# ---------------------------------------------------------

GFF_FILE = "genomic.gff"

FASTA_FILE = "ref.fna"


# ---------------------------------------------------------
# OUTPUT FILES
# ---------------------------------------------------------

OUTPUT_FASTA = "T_atroviride_promoters.fasta"

OUTPUT_TABLE = "T_atroviride_promoters.tsv"


# ---------------------------------------------------------
# PROMOTER LENGTH
# ---------------------------------------------------------

# Maximum distance upstream of the transcription start region.

# NOTE:
# At this stage we are using the annotated gene boundary as
# a proxy for the promoter reference point.
#
# Later, if you have TSS information, this can be changed
# to use experimentally predicted TSS positions.

UPSTREAM_LENGTH = 1000


# ---------------------------------------------------------
# MINIMUM PROMOTER LENGTH
# ---------------------------------------------------------

# Promoters shorter than this will not be exported.

MIN_PROMOTER_LENGTH = 999


# ---------------------------------------------------------
# USE NEIGHBORING GENES AS A BOUNDARY?
# ---------------------------------------------------------

# True:
#   Do not allow the promoter to extend through another
#   annotated gene.
#
# False:
#   Always extract exactly UPSTREAM_LENGTH where possible.

AVOID_UPSTREAM_GENES = True


# ---------------------------------------------------------
# ONLY CHROMOSOME NC_089400.1
# ---------------------------------------------------------

# Your current project is chromosome 1.

TARGET_SEQUENCE = "NC_089400.1"


# =========================================================
# IMPORTS
# =========================================================

import re
from collections import defaultdict


# =========================================================
# FASTA READER
# =========================================================

def read_fasta(filename):

    sequences = {}

    current_id = None
    sequence_parts = []

    print("\nReading FASTA:")
    print(filename)

    with open(filename, "r") as file:

        for line in file:

            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):

                # Save previous sequence
                if current_id is not None:

                    sequences[current_id] = "".join(
                        sequence_parts
                    ).upper()

                # FASTA identifier
                current_id = line[1:].split()[0]

                sequence_parts = []

            else:

                sequence_parts.append(line)

        # Save final sequence
        if current_id is not None:

            sequences[current_id] = "".join(
                sequence_parts
            ).upper()

    print(
        f"Loaded {len(sequences)} FASTA sequence(s)."
    )

    for seq_id, sequence in sequences.items():

        print(
            f"  {seq_id}: {len(sequence):,} bp"
        )

    return sequences


# =========================================================
# REVERSE COMPLEMENT
# =========================================================

def reverse_complement(sequence):

    complement = str.maketrans(
        "ACGTNacgtn",
        "TGCANtgcan"
    )

    return sequence.translate(complement)[::-1]


# =========================================================
# GFF3 ATTRIBUTE PARSER
# =========================================================

def parse_attributes(attribute_text):

    attributes = {}

    if attribute_text == ".":

        return attributes

    for item in attribute_text.split(";"):

        item = item.strip()

        if not item:
            continue

        if "=" in item:

            key, value = item.split(
                "=",
                1
            )

            attributes[key] = value

        else:

            # Some GFF files can contain
            # space-separated attributes.

            parts = item.split(
                None,
                1
            )

            if len(parts) == 2:

                attributes[
                    parts[0]
                ] = parts[1].strip('"')

    return attributes


# =========================================================
# READ GFF3
# =========================================================

def read_gff(filename):

    print("\nReading GFF3:")
    print(filename)

    genes = []

    with open(filename, "r") as file:

        for line in file:

            line = line.strip()

            # Skip comments
            if not line:
                continue

            if line.startswith("#"):
                continue

            fields = line.split("\t")

            if len(fields) != 9:
                continue

            seqid = fields[0]
            feature = fields[2]

            try:

                start = int(fields[3])
                end = int(fields[4])

            except ValueError:

                continue

            strand = fields[6]

            attributes = parse_attributes(
                fields[8]
            )

            # -------------------------------------------------
            # IMPORTANT
            # -------------------------------------------------
            #
            # We want ONE record per gene.
            #
            # NCBI GFF files can contain:
            #
            # gene
            # mRNA
            # CDS
            # exon
            #
            # We therefore prioritize "gene".
            #

            if feature != "gene":
                continue

            # Try several possible NCBI identifiers.

            gene_id = (
                attributes.get("locus_tag")
                or attributes.get("gene")
                or attributes.get("Name")
                or attributes.get("ID")
            )

            if gene_id is None:

                gene_id = (
                    f"{seqid}:"
                    f"{start}-"
                    f"{end}"
                )

            genes.append({

                "seqid": seqid,

                "start": start,

                "end": end,

                "strand": strand,

                "gene_id": gene_id,

                "attributes": attributes

            })

    print(
        f"Found {len(genes):,} gene records."
    )

    return genes


# =========================================================
# FIND CLOSEST UPSTREAM GENE
# =========================================================

def find_closest_upstream_gene(
    gene,
    genes_on_sequence
):

    start = gene["start"]
    end = gene["end"]

    strand = gene["strand"]

    closest_gene = None

    closest_distance = None

    for other in genes_on_sequence:

        if other is gene:
            continue

        # ---------------------------------------------
        # PLUS STRAND
        # ---------------------------------------------

        if strand == "+":

            # Upstream genes have smaller coordinates.

            if other["end"] < start:

                distance = (
                    start -
                    other["end"]
                )

                if (
                    closest_distance is None
                    or distance < closest_distance
                ):

                    closest_distance = distance

                    closest_gene = other

        # ---------------------------------------------
        # MINUS STRAND
        # ---------------------------------------------

        elif strand == "-":

            # Upstream genes have larger coordinates.

            if other["start"] > end:

                distance = (
                    other["start"] -
                    end
                )

                if (
                    closest_distance is None
                    or distance < closest_distance
                ):

                    closest_distance = distance

                    closest_gene = other

    return closest_gene, closest_distance


# =========================================================
# EXTRACT PROMOTER
# =========================================================

def extract_promoter(
    gene,
    genome_sequence,
    genes_on_sequence
):

    start = gene["start"]

    end = gene["end"]

    strand = gene["strand"]

    chromosome_length = len(
        genome_sequence
    )

    # =====================================================
    # PLUS STRAND
    # =====================================================

    if strand == "+":

        # Gene begins at START.
        #
        # Candidate promoter:
        #
        # START-1000 ... START-1

        promoter_end = start - 1

        promoter_start = max(
            1,
            start - UPSTREAM_LENGTH
        )

        # ---------------------------------------------
        # Check upstream gene
        # ---------------------------------------------

        if AVOID_UPSTREAM_GENES:

            upstream_gene, distance = (
                find_closest_upstream_gene(
                    gene,
                    genes_on_sequence
                )
            )

            if upstream_gene is not None:

                # Leave one base between
                # promoter and upstream gene.

                maximum_length = max(
                    1,
                    distance - 1
                )

                actual_length = min(
                    UPSTREAM_LENGTH,
                    maximum_length
                )

                promoter_start = (
                    start -
                    actual_length
                )

        # Python uses 0-based coordinates.

        sequence = genome_sequence[
            promoter_start - 1 :
            promoter_end
        ]


    # =====================================================
    # MINUS STRAND
    # =====================================================

    elif strand == "-":

        # For a minus-strand gene, transcription proceeds
        # toward decreasing genomic coordinates.
        #
        # Therefore the promoter is AFTER the gene in
        # genomic coordinates.

        promoter_start = end + 1

        promoter_end = min(
            chromosome_length,
            end + UPSTREAM_LENGTH
        )

        # ---------------------------------------------
        # Check upstream gene
        # ---------------------------------------------

        if AVOID_UPSTREAM_GENES:

            upstream_gene, distance = (
                find_closest_upstream_gene(
                    gene,
                    genes_on_sequence
                )
            )

            if upstream_gene is not None:

                maximum_length = max(
                    1,
                    distance - 1
                )

                actual_length = min(
                    UPSTREAM_LENGTH,
                    maximum_length
                )

                promoter_end = (
                    end +
                    actual_length
                )

        sequence = genome_sequence[
            promoter_start - 1 :
            promoter_end
        ]

        # IMPORTANT:
        #
        # Convert to gene orientation:
        #
        # genomic 3' -> 5'
        #
        # becomes
        #
        # promoter 5' -> 3'

        sequence = reverse_complement(
            sequence
        )

    else:

        return None


    # =====================================================
    # VALIDATE
    # =====================================================

    if len(sequence) < MIN_PROMOTER_LENGTH:

        return None


    return {

        "sequence": sequence,

        "start": promoter_start,

        "end": promoter_end,

        "length": len(sequence)

    }


# =========================================================
# WRITE FASTA
# =========================================================

def write_fasta_record(
    file,
    header,
    sequence
):

    file.write(
        header + "\n"
    )

    # Wrap sequence at 80 bp.

    for i in range(
        0,
        len(sequence),
        80
    ):

        file.write(
            sequence[i:i+80] + "\n"
        )


# =========================================================
# MAIN ANALYSIS
# =========================================================

def main():

    print("=" * 70)

    print(
        "Trichoderma atroviride promoter extraction"
    )

    print("=" * 70)

    print(
        f"\nTarget chromosome: {TARGET_SEQUENCE}"
    )

    print(
        f"Maximum promoter length: "
        f"{UPSTREAM_LENGTH} bp"
    )

    # =====================================================
    # LOAD GENOME
    # =====================================================

    genome = read_fasta(
        FASTA_FILE
    )

    # Check chromosome.

    if TARGET_SEQUENCE not in genome:

        print(
            "\nERROR:"
        )

        print(
            f"Could not find {TARGET_SEQUENCE} "
            "in the FASTA file."
        )

        print(
            "\nAvailable sequences:"
        )

        for seq_id in genome:

            print(
                f"  {seq_id}"
            )

        return

    chromosome = genome[
        TARGET_SEQUENCE
    ]

    # =====================================================
    # LOAD GFF
    # =====================================================

    genes = read_gff(
        GFF_FILE
    )

    # =====================================================
    # ONLY TARGET CHROMOSOME
    # =====================================================

    genes = [

        gene

        for gene in genes

        if gene["seqid"] == TARGET_SEQUENCE

    ]

    print(
        f"\nGenes on {TARGET_SEQUENCE}: "
        f"{len(genes):,}"
    )

    # =====================================================
    # SORT GENES
    # =====================================================

    genes.sort(
        key=lambda x: x["start"]
    )

    # =====================================================
    # OUTPUT FILES
    # =====================================================

    fasta_file = open(
        OUTPUT_FASTA,
        "w"
    )

    table_file = open(
        OUTPUT_TABLE,
        "w"
    )

    # =====================================================
    # TABLE HEADER
    # =====================================================

    table_file.write(
        "gene_id\t"
        "seqid\t"
        "strand\t"
        "gene_start\t"
        "gene_end\t"
        "promoter_start\t"
        "promoter_end\t"
        "promoter_length\t"
        "promoter_sequence\n"
    )

    # =====================================================
    # EXTRACT PROMOTERS
    # =====================================================

    extracted = 0

    skipped = 0

    for gene in genes:

        promoter = extract_promoter(

            gene,

            chromosome,

            genes

        )

        if promoter is None:

            skipped += 1

            continue

        sequence = promoter[
            "sequence"
        ]

        gene_id = gene[
            "gene_id"
        ]

        # Clean ID for FASTA.

        clean_id = re.sub(
            r"[^A-Za-z0-9_.-]",
            "_",
            str(gene_id)
        )

        # =================================================
        # FASTA HEADER
        # =================================================

        header = (

            f">{clean_id}"

            f"|chromosome={TARGET_SEQUENCE}"

            f"|strand={gene['strand']}"

            f"|gene="
            f"{gene['start']}-"
            f"{gene['end']}"

            f"|promoter="
            f"{promoter['start']}-"
            f"{promoter['end']}"

            f"|length="
            f"{promoter['length']}"

        )

        write_fasta_record(

            fasta_file,

            header,

            sequence

        )

        # =================================================
        # TABLE
        # =================================================

        table_file.write(

            f"{gene_id}\t"

            f"{TARGET_SEQUENCE}\t"

            f"{gene['strand']}\t"

            f"{gene['start']}\t"

            f"{gene['end']}\t"

            f"{promoter['start']}\t"

            f"{promoter['end']}\t"

            f"{promoter['length']}\t"

            f"{sequence}\n"

        )

        extracted += 1

    # =====================================================
    # CLOSE FILES
    # =====================================================

    fasta_file.close()

    table_file.close()

    # =====================================================
    # SUMMARY
    # =====================================================

    print("\n" + "=" * 70)

    print("FINISHED")

    print("=" * 70)

    print(
        f"\nGenes processed:   {len(genes):,}"
    )

    print(
        f"Promoters saved:   {extracted:,}"
    )

    print(
        f"Genes skipped:     {skipped:,}"
    )

    print(
        f"\nFASTA output:"
        f"\n  {OUTPUT_FASTA}"
    )

    print(
        f"\nTable output:"
        f"\n  {OUTPUT_TABLE}"
    )

    print("\n" + "=" * 70)

    print(
        "You can now use the FASTA file for "
        "promoter/TSS motif analysis."
    )

    print("=" * 70)


# =========================================================
# RUN
# =========================================================

if __name__ == "__main__":

    main()



Trichoderma atroviride promoter extraction

Target chromosome: NC_089400.1
Maximum promoter length: 1000 bp

Reading FASTA:
ref.fna
Loaded 7 FASTA sequence(s).
  NC_089400.1: 7,285,777 bp
  NC_089401.1: 7,088,962 bp
  NC_089402.1: 5,658,044 bp
  NC_089403.1: 5,267,733 bp
  NC_089404.1: 4,072,028 bp
  NC_089405.1: 4,041,281 bp
  NC_089406.1: 3,886,821 bp

Reading GFF3:
genomic.gff
Found 13,312 gene records.

Genes on NC_089400.1: 2,645

FINISHED

Genes processed:   2,645
Promoters saved:   1,110
Genes skipped:     1,535

FASTA output:
  T_atroviride_promoters.fasta

Table output:
  T_atroviride_promoters.tsv

You can now use the FASTA file for promoter/TSS motif analysis.
